# Visibility Thresholds: independent judges in Colab

Run this notebook in a **separate Colab GPU runtime**, never inside the CLIPDraw runtime. It reads completed Drive images only. Each selected judge runs in its own managed Python 3.12.10 subprocess environment. The kernel keeps its original Python/PyTorch packages. Florence uses Transformers 4.49.0 custom code pinned to the model commit; SigLIP2 uses 5.3.0. A CUDA 12.4 / Torch 2.6 / torchvision 0.21 pair is installed only in those environments. T4/L4/A100-class GPUs are the conservative target; preflight stops on unsupported allocations/drivers.

Before upload, run `python -m judges.prepare_colab --output judge_results/visibility_judges_source.zip` locally. Upload that archive and this notebook to your Drive/Colab manually. The archive includes uncommitted source and the vocabulary, not generated images or weights. No push is required. Full instructions: `judges/COLAB.md`.

Default mode is **benchmark**. Review configuration before running. Paid access does not guarantee a particular GPU, speed or session duration.


## 1. Shared configuration

Edit paths and your stable job name here. Point INPUT_ROOTS at complete team part folders, omitting smoke-test outputs. Each writer uses a separate shard/checkpoint. Freeze the input cohort for a production job. Use the same configuration, package locks and GPU type to resume; changed conditions must use a new job name. MODEL selects one judge at a time. For a new benchmark measurement, change BENCHMARK_TAG; benchmarks cannot resume.


In [ ]:
from pathlib import Path

MODEL = "siglip2"  # "siglip2" or "florence2"
MODE = "benchmark"  # "benchmark" or "run"
RESUME = False  # True only for an interrupted production job.
SOURCE_ARCHIVE = Path("/content/drive/MyDrive/visibility_judges_source.zip")
LOCAL_SOURCE = Path("/content/visibility_judges")
INPUT_ROOTS = [
    "/content/drive/MyDrive/clipdraw_results/part1",
    "/content/drive/MyDrive/clipdraw_results/part2",
]
OUTPUT_ROOT = Path("/content/drive/MyDrive/visibility_judge_results")
JOB_NAME = "pilot35_v1"
SHARD_INDEX = 1  # 1-based; teammates use distinct shards under the same cohort.
NUM_SHARDS = 1
SIGLIP2_BATCH_SIZE = 2
FLORENCE2_BATCH_SIZE = 1
SIGLIP2_REVISION = "75de2d55ec2d0b4efc50b3e9ad70dba96a7b2fa2"
FLORENCE2_REVISION = "4a12a2b54b7016a48a22037fbd62da90cd566f2a"
SIGLIP2_PROMPT = "A drawing of a {}."
TEXT_BATCH_SIZE = 32
FLORENCE2_DTYPE = "auto"  # float16 on CUDA; choose float32 for a separate condition.
MAX_NEW_TOKENS = 128
NUM_BEAMS = 3
BENCHMARK_TAG = "bench_t4_01"
BENCHMARK_IMAGES = 8
WARMUP_IMAGES = 1
ESTIMATE_SIZES = [150, 2000, 2500]
# Disk cache is separate from checkpoints. /content is temporary; use a Drive
# folder instead if you want model downloads to survive runtime deletion.
MODEL_CACHE = "/content/hf_judge_cache"
# Optional comparison across completed, nonoverlapping shard/session exports.
SIGLIP2_EXPORTS = []
FLORENCE2_EXPORTS = []
ANALYSIS_DIR = OUTPUT_ROOT / "analysis_pilot35_v1"  # Must be new.

if MODEL not in {"siglip2", "florence2"} or MODE not in {"benchmark", "run"}:
    raise ValueError("Select a supported MODEL and MODE.")
if MODE == "benchmark" and RESUME:
    raise ValueError("Benchmarks need fresh measurements; use a new BENCHMARK_TAG.")


## 2. Mount Drive

Add shared team folders as shortcuts in My Drive before mounting. Keep completed source images/metadata read only. Checkpoints, environment locks and exports go directly to OUTPUT_ROOT in Drive so they persist across runtime loss. Filesystem atomic-replace behavior still needs real Colab/Drive validation.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 3. Verify and unpack the local source bundle

This reuses an identical extracted copy. If the source differs, use a fresh runtime/local directory and a new production job; never silently mix source versions during resume.


In [ ]:
import hashlib
import json
import os
import sys
import zipfile
import subprocess

if not os.path.ismount("/content/drive"):
    raise RuntimeError("Mount Drive first.")
if not OUTPUT_ROOT.resolve().is_relative_to(Path("/content/drive")):
    raise ValueError("OUTPUT_ROOT must be on mounted Drive for persistent checkpoints.")
bundle_hash = hashlib.sha256(SOURCE_ARCHIVE.read_bytes()).hexdigest()
marker = LOCAL_SOURCE / ".bundle.sha256"
if LOCAL_SOURCE.exists() and any(LOCAL_SOURCE.iterdir()):
    if not marker.is_file() or marker.read_text().strip() != bundle_hash:
        raise ValueError("Existing local source is from a different/unverified bundle. Use a fresh runtime/local directory.")
with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
    manifest = json.loads(archive.read("source_manifest.json"))
    if set(archive.namelist()) != set(manifest) | {"source_manifest.json"}:
        raise ValueError("Bundle entries disagree with source manifest.")
    for name, expected in manifest.items():
        relative = Path(name)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("Unsafe archive path.")
        content = archive.read(name)
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError(f"Source bundle checksum mismatch: {name}")
        target = LOCAL_SOURCE / relative
        if target.exists() and target.read_bytes() != content:
            raise ValueError(f"Existing source differs: {target}. Use a fresh runtime/local path.")
    for name in manifest:
        target = LOCAL_SOURCE / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            target.write_bytes(archive.read(name))
marker.write_text(bundle_hash, encoding="utf-8")
print("Verified source files:", len(manifest))
subprocess.run(["nvidia-smi"], check=True)


## 4. Build selected judge environment and run CUDA preflight

This may download a managed interpreter, CUDA wheels and dependencies. Only the small uv launcher is installed in the kernel. Package freezes are saved in Drive and replayed on reconnect; differing freezes/configurations stop the job. The same shared OUTPUT_ROOT provides the same per-model dependency lock for team sessions. Setup does not load model weights.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.13.5"], check=True)
subprocess.run([
    sys.executable, "-m", "judges.colab_setup", "--model", MODEL,
    "--lock-root", str(OUTPUT_ROOT / "environment_locks"),
], cwd=LOCAL_SOURCE, check=True)
JUDGE_PYTHON = Path("/content/judge_envs") / MODEL / "bin/python"


## 5. Inspect the cohort and run benchmark or production

A benchmark warms the model, measures a small fresh sample, and records model loading separately from inference and checkpoint IO. It estimates 150, 2,000 and the configured larger workloads; it does not claim guaranteed paid-Colab speed. Benchmark outputs are separate from production.

After checking the benchmark, choose MODE="run", keep batch size conservative, and rerun configuration and this cell. After a disconnect, remount Drive, restore the same source/configuration/environment, select MODE="run" and RESUME=True, then rerun setup and this cell. Successfully checkpointed images are skipped. Source, vocabulary, cohort or inference changes are rejected rather than reusing stale predictions.


In [ ]:
configuration = {
    "model": MODEL, "revision": SIGLIP2_REVISION if MODEL == "siglip2" else FLORENCE2_REVISION,
    "input_roots": INPUT_ROOTS, "output_root": str(OUTPUT_ROOT), "job_name": JOB_NAME,
    "words_source": str(LOCAL_SOURCE / "dataset/processed/sampled_words.csv"),
    "device": "cuda", "batch_size": SIGLIP2_BATCH_SIZE if MODEL == "siglip2" else FLORENCE2_BATCH_SIZE,
    "shard_index": SHARD_INDEX, "num_shards": NUM_SHARDS,
    "benchmark_tag": BENCHMARK_TAG, "benchmark_images": BENCHMARK_IMAGES,
    "warmup_images": WARMUP_IMAGES, "estimate_sizes": ESTIMATE_SIZES,
    "cache_dir": MODEL_CACHE,
    "source_bundle_sha256": hashlib.sha256(SOURCE_ARCHIVE.read_bytes()).hexdigest(),
}
if MODEL == "siglip2":
    configuration.update(prompt_template=SIGLIP2_PROMPT, text_batch_size=TEXT_BATCH_SIZE)
else:
    configuration.update(dtype=FLORENCE2_DTYPE, max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
config_path = LOCAL_SOURCE / "colab_job.json"
config_path.write_text(json.dumps(configuration, indent=2), encoding="utf-8")
print(json.dumps(configuration, indent=2))
command = [str(JUDGE_PYTHON), "-m", "judges.scaling", "--config", str(config_path), "--mode", MODE]
if RESUME:
    command.append("--resume")
subprocess.run(command, cwd=LOCAL_SOURCE, check=True)


## 6. Compare completed exports from separate sessions

Populate SIGLIP2_EXPORTS and FLORENCE2_EXPORTS in configuration with the full/results.json paths from all completed, nonoverlapping shards. Model conditions must match within each judge. Do not include both an early cumulative export and its resumed replacement, or mix benchmark samples with full exports. Analysis verifies image IDs/content and rejects duplicate/inconsistent inputs. Captions and heuristic text mentions are model descriptions, not human recognition or verified OCR.


In [ ]:
if SIGLIP2_EXPORTS and FLORENCE2_EXPORTS:
    subprocess.run([
        sys.executable, "-m", "judges.analyze", "--siglip2", *SIGLIP2_EXPORTS,
        "--florence2", *FLORENCE2_EXPORTS, "--source-root", *INPUT_ROOTS,
        "--output-dir", str(ANALYSIS_DIR),
    ], cwd=LOCAL_SOURCE, check=True)
else:
    print("Analysis skipped: configure both export lists when all selected jobs finish.")


## Finish

Disconnect/delete the GPU runtime when done. Each model subprocess releases its GPU memory on exit, so switching MODEL does not retain the other model's weights. Only Drive artifacts survive a deleted runtime; source bundles, checkpoint directories and environment locks should be retained together. Actual Colab CUDA, Drive atomic replacement/resume behavior and throughput must be validated with the small benchmark before a 2,000+ image run.
